# CMS Hospital Readmission Risk Analysis
## Notebook 3 — Feature Engineering

**Input:**  `data/clean/cms_patients_clean.csv`  
**Output:** `data/engineered/cms_patients_engineered.csv` + `data/engineered/label_encoders.json`

---

### What this notebook does
Transforms the 36-column clean dataset into a 72-column model-ready feature set across 5 groups:

| Group | Features | Description |
|-------|----------|-------------|
| 1 — Temporal | 9 | Year, month, quarter, day of week, season, weekend flag |
| 2 — Demographic bands | 7 | Age group, LOS band, cost band, BMI category (WHO), med band |
| 3 — Binary clinical flags | 13 | is_elderly, is_polypharmacy, is_frequent_flyer, etc. |
| 4 — Composite scores | 4 | composite_risk_flags, social_risk_score, cost_per_day |
| 5 — Encoded categoricals | 15 | Label-encoded integer codes for all categorical columns |

In [18]:
import os

# Set working directory
os.chdir(r"C:\Users\abelo\Desktop\cms_readmission")

# Create all folders
os.makedirs('data/raw',        exist_ok=True)
os.makedirs('data/clean',      exist_ok=True)
os.makedirs('data/engineered', exist_ok=True)
os.makedirs('data/outputs',    exist_ok=True)
os.makedirs('reports',         exist_ok=True)

print('Working directory:', os.getcwd())
print('Folders ready.')

Working directory: C:\Users\abelo\Desktop\cms_readmission
Folders ready.


### 3.1 — Import libraries

In [19]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder

print('Libraries loaded successfully')

Libraries loaded successfully


### 3.2 — Load clean data

In [11]:
df = pd.read_csv(r"C:\Users\abelo\Desktop\cms_readmission\raw\cms_patients_raw.csv")
df['admit_date']     = pd.to_datetime(df['admit_date'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])

print(f'Input: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Nulls: {df.isnull().sum().sum()}')

Input: 5,000 rows × 20 columns
Nulls: 200


### 3.3 — Group 1: Temporal features

In [12]:
df['admit_year']       = df['admit_date'].dt.year
df['admit_month']      = df['admit_date'].dt.month
df['admit_month_name'] = df['admit_date'].dt.strftime('%b')
df['admit_quarter']    = df['admit_date'].dt.quarter
df['admit_dow']        = df['admit_date'].dt.day_name()
df['admit_week']       = df['admit_date'].dt.isocalendar().week.astype(int)
df['is_weekend_admit'] = df['admit_date'].dt.dayofweek.isin([5, 6]).astype(int)
df['discharge_month']  = df['discharge_date'].dt.month
df['season']           = df['admit_month'].map({
    12:'Winter',1:'Winter',2:'Winter',
    3:'Spring',4:'Spring',5:'Spring',
    6:'Summer',7:'Summer',8:'Summer',
    9:'Autumn',10:'Autumn',11:'Autumn',
})

print('Group 1 — Temporal features: 9 features created')
print(f'  Weekend admissions: {df["is_weekend_admit"].sum():,} ({df["is_weekend_admit"].mean():.1%})')
print(f'  Season distribution:')
print(df['season'].value_counts().to_string())

Group 1 — Temporal features: 9 features created
  Weekend admissions: 1,428 (28.6%)
  Season distribution:
season
Winter    1700
Spring    1104
Summer    1104
Autumn    1092


### 3.4 — Group 2: Demographic banding

In [13]:
df['age_group'] = pd.cut(df['age'],
    bins=[0,39,54,64,74,84,120],
    labels=['18-39','40-54','55-64','65-74','75-84','85+'])

df['los_band'] = pd.cut(df['length_of_stay_days'],
    bins=[0,2,4,7,11,999],
    labels=['1-2d','3-4d','5-7d','8-11d','12+d'])

df['cost_band'] = pd.cut(df['total_cost_usd'],
    bins=[0,14999,24999,39999,999999],
    labels=['<$15K','$15K-$25K','$25K-$40K','>$40K'])

df['med_band'] = pd.cut(df['num_medications'],
    bins=[-1,3,7,11,15,999],
    labels=['0-3','4-7','8-11','12-15','16+'])

df['comorbidity_band'] = pd.cut(df['comorbidity_index'],
    bins=[-0.1,2,4,6,8,11],
    labels=['0-2','2-4','4-6','6-8','8-10'])

df['prior_admit_band'] = pd.cut(df['prior_admissions_12mo'],
    bins=[-1,0,1,2,4,99],
    labels=['0','1','2','3-4','5+'])

df['bmi_category'] = pd.cut(df['bmi'],
    bins=[0,18.5,24.9,29.9,34.9,999],
    labels=['Underweight','Normal','Overweight','Obese I','Obese II+'])

print('Group 2 — Demographic bands: 7 features created')
print('\nAge group distribution:')
print(df['age_group'].value_counts().sort_index().to_string())
print('\nBMI category distribution:')
print(df['bmi_category'].value_counts().to_string())

Group 2 — Demographic bands: 7 features created

Age group distribution:
age_group
18-39     117
40-54     748
55-64    1174
65-74    1390
75-84     996
85+       550

BMI category distribution:
bmi_category
Overweight     1419
Obese I        1231
Normal         1149
Obese II+       838
Underweight     338


### 3.5 — Group 3: Binary clinical flags

In [14]:
df['is_elderly']          = (df['age'] >= 75).astype(int)
df['is_high_comorbidity'] = (df['comorbidity_index'] >= 5).astype(int)
df['is_polypharmacy']     = (df['num_medications'] >= 10).astype(int)
df['is_frequent_flyer']   = (df['prior_admissions_12mo'] >= 3).astype(int)
df['is_long_stay']        = (df['length_of_stay_days'] >= 8).astype(int)
df['is_self_pay']         = (df['payer'] == 'Self-Pay').astype(int)
df['is_ama_discharge']    = (df['discharge_disposition'].str.lower() == 'ama').astype(int)
df['is_home_discharge']   = (df['discharge_disposition'] == 'Home').astype(int)
df['is_obese']            = (df['bmi'] >= 30).astype(int)
df['is_underweight']      = (df['bmi'] < 18.5).astype(int)
df['is_high_ed_user']     = (df['ed_visits_prior_6mo'] >= 3).astype(int)
df['is_medicare']         = (df['payer'] == 'Medicare').astype(int)
df['is_medicaid']         = (df['payer'] == 'Medicaid').astype(int)

print('Group 3 — Binary clinical flags: 13 features created')
flags = ['is_elderly','is_high_comorbidity','is_polypharmacy','is_frequent_flyer',
         'is_long_stay','is_self_pay','is_high_ed_user']
for f in flags:
    print(f'  {f:<25} {df[f].sum():>5,} patients ({df[f].mean():.1%})')

Group 3 — Binary clinical flags: 13 features created
  is_elderly                1,546 patients (30.9%)
  is_high_comorbidity         416 patients (8.3%)
  is_polypharmacy           1,430 patients (28.6%)
  is_frequent_flyer           846 patients (16.9%)
  is_long_stay              1,285 patients (25.7%)
  is_self_pay                 315 patients (6.3%)
  is_high_ed_user             564 patients (11.3%)


### 3.6 — Group 4: Composite risk scores

In [15]:
df['composite_risk_flags'] = (
    df['is_elderly'] + df['is_high_comorbidity'] + df['is_polypharmacy']
    + df['is_frequent_flyer'] + df['is_long_stay']
)
df['extended_risk_score'] = (
    df['composite_risk_flags']
    + df['is_self_pay'] + df['is_high_ed_user'] + df['is_ama_discharge']
)
df['social_risk_score'] = (
    (df['age'] > 75).astype(float) * 0.3
    + (df['prior_admissions_12mo'] > 2).astype(float) * 0.4
    + (df['comorbidity_index'] > 5).astype(float) * 0.3
    + df['ed_visits_prior_6mo'] * 0.05
).round(3)
df['cost_per_day'] = (df['total_cost_usd'] / df['length_of_stay_days']
                     ).round(0).fillna(0).astype(int)

print('Group 4 — Composite risk scores: 4 features created')
print(f'  composite_risk_flags distribution (0–5):')
print(df['composite_risk_flags'].value_counts().sort_index().to_string())

Group 4 — Composite risk scores: 4 features created
  composite_risk_flags distribution (0–5):
composite_risk_flags
0    1399
1    2051
2    1223
3     284
4      41
5       2


### 3.7 — Group 5: Label encode categoricals

In [16]:
cat_cols = [
    'primary_diagnosis','discharge_disposition','payer',
    'gender','ethnicity','hospital','state','season',
    'age_group','los_band','cost_band','med_band',
    'comorbidity_band','prior_admit_band','bmi_category',
]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = list(le.classes_)

print(f'Group 5 — Encoded categoricals: {len(cat_cols)} features created')
print('\nEncoding maps saved (sample):')
for col in ['primary_diagnosis','payer','model_risk_tier'] if 'model_risk_tier' in encoders else ['primary_diagnosis','payer']:
    if col in encoders:
        print(f'  {col}: {encoders[col]}')

Group 5 — Encoded categoricals: 15 features created

Encoding maps saved (sample):
  primary_diagnosis: ['Acute MI', 'CABG', 'COPD', 'Diabetes', 'Heart Failure', 'Hip/Knee Replacement', 'Pneumonia', 'Renal Failure', 'Sepsis', 'Stroke']
  payer: ['Medicaid', 'Medicare', 'Private Insurance', 'Self-Pay']


### 3.8 — Save engineered dataset

In [17]:
# Convert pandas categoricals to string for CSV
band_cols = ['age_group','los_band','cost_band','med_band',
             'comorbidity_band','prior_admit_band','bmi_category']
for col in band_cols:
    df[col] = df[col].astype(str)

df['admit_date']     = df['admit_date'].dt.strftime('%Y-%m-%d')
df['discharge_date'] = df['discharge_date'].dt.strftime('%Y-%m-%d')

os.makedirs('data/engineered', exist_ok=True)
df.to_csv('data/engineered/cms_patients_engineered.csv', index=False)

with open('data/engineered/label_encoders.json', 'w') as f:
    json.dump(encoders, f, indent=2)

print(f'Output: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'New features added: {df.shape[1] - 36}')
print(f'\nFiles saved:')
print(f'  data/engineered/cms_patients_engineered.csv')
print(f'  data/engineered/label_encoders.json')

Output: 5,000 rows × 67 columns
New features added: 31

Files saved:
  data/engineered/cms_patients_engineered.csv
  data/engineered/label_encoders.json


---
**Notebook 3 complete.** Output saved to `data/engineered/cms_patients_engineered.csv`

Next → **Notebook 4: Predictive Modelling**